In [1]:
from datetime import datetime
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler


In [2]:
# rutas base
REPO_ROOT = Path('.').resolve()
RUN_ID = datetime.today().strftime('%Y%m%d')
INPUT_DIR = REPO_ROOT / 'data' / 'intermediate' / '02_train_test_v2' / RUN_ID
OUTPUT_DIR = REPO_ROOT / 'data' / 'intermediate' / '03_normalizacion_v2' / RUN_ID

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_DIRS = ['1_vs_resto', '5_vs_resto']

if not INPUT_DIR.exists():
    candidates = sorted((REPO_ROOT / 'data' / 'intermediate' / '02_train_test_v2').glob('*/1_vs_resto'))
    if candidates:
        INPUT_DIR = candidates[-1].parent
        print(f'Usando ultimo split encontrado: {INPUT_DIR}')
    else:
        raise FileNotFoundError('No se encontro data/intermediate/02_train_test_v2 con ramas 1_vs_resto y 5_vs_resto')

print('Rutas configuradas:')
print(f'INPUT_DIR: {INPUT_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'TARGET_DIRS: {TARGET_DIRS}')


Rutas configuradas:
INPUT_DIR: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\02_train_test_v2\20260329
OUTPUT_DIR: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\03_normalizacion_v2\20260329
TARGET_DIRS: ['1_vs_resto', '5_vs_resto']


In [3]:
scalers = {
    'Standard': StandardScaler(),
    'MinMax': MinMaxScaler(),
    'MaxAbs': MaxAbsScaler(),
    'Robust': RobustScaler(),
    'NoNorm': None
}

resumen_global = []


In [4]:
for target_name in TARGET_DIRS:
    input_subdir = INPUT_DIR / target_name
    output_subdir = OUTPUT_DIR / target_name
    output_subdir.mkdir(parents=True, exist_ok=True)

    x_train_file = input_subdir / 'X_train.parquet'
    x_test_file = input_subdir / 'X_test.parquet'
    y_train_file = input_subdir / 'y_train.parquet'
    y_test_file = input_subdir / 'y_test.parquet'

    if not x_train_file.exists() or not x_test_file.exists():
        print(f'No se encontro split para {target_name}; se omite.')
        continue

    X_train = pd.read_parquet(x_train_file)
    X_test = pd.read_parquet(x_test_file)
    y_train = pd.read_parquet(y_train_file)
    y_test = pd.read_parquet(y_test_file)

    original_n_columns = X_train.shape[1]
    print(f'\nProcesando {target_name}')
    print(f'Columnas originales (antes de normalizar): {original_n_columns}')

    metricas = []

    for name, scaler in scalers.items():
        norm_dir = output_subdir / name
        norm_dir.mkdir(parents=True, exist_ok=True)

        if scaler:
            scaler.fit(X_train)
            X_train_norm = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
            X_test_norm = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
        else:
            X_train_norm = X_train.copy()
            X_test_norm = X_test.copy()

        X_train_norm.to_parquet(norm_dir / f'X_train_{name}.parquet')
        X_test_norm.to_parquet(norm_dir / f'X_test_{name}.parquet')
        y_train.to_parquet(norm_dir / f'y_train_{name}.parquet')
        y_test.to_parquet(norm_dir / f'y_test_{name}.parquet')

        n_cols_train = X_train_norm.shape[1]
        print(f'{target_name} | {name}: columnas despues de normalizar = {n_cols_train}')

        fila = {
            'target': target_name,
            'normalizacion': name,
            'columnas_originales': original_n_columns,
            'columnas_resultantes': n_cols_train
        }
        metricas.append(fila)
        resumen_global.append(fila)

    df_metricas = pd.DataFrame(metricas)
    df_metricas.to_csv(output_subdir / 'resumen_columnas_normalizacion.csv', index=False)
    print(f'Resumen guardado en: {output_subdir / "resumen_columnas_normalizacion.csv"}')

if resumen_global:
    df_resumen_global = pd.DataFrame(resumen_global)
    df_resumen_global.to_csv(OUTPUT_DIR / 'resumen_columnas_normalizacion_global.csv', index=False)
    print(f'\nResumen global guardado en: {OUTPUT_DIR / "resumen_columnas_normalizacion_global.csv"}')
else:
    print('No se generaron normalizaciones; revisa las rutas de entrada.')



Procesando 1_vs_resto
Columnas originales (antes de normalizar): 571
1_vs_resto | Standard: columnas despues de normalizar = 571
1_vs_resto | MinMax: columnas despues de normalizar = 571
1_vs_resto | MaxAbs: columnas despues de normalizar = 571
1_vs_resto | Robust: columnas despues de normalizar = 571
1_vs_resto | NoNorm: columnas despues de normalizar = 571
Resumen guardado en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\03_normalizacion_v2\20260329\1_vs_resto\resumen_columnas_normalizacion.csv

Procesando 5_vs_resto
Columnas originales (antes de normalizar): 571
5_vs_resto | Standard: columnas despues de normalizar = 571
5_vs_resto | MinMax: columnas despues de normalizar = 571
5_vs_resto | MaxAbs: columnas despues de normalizar = 571
5_vs_resto | Robust: columnas despues de normalizar = 571
5_vs_resto | NoNorm: columnas despues de normalizar = 571
Resumen guardado en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermedia